## 涨停分析：撰写特征，从涨到threshold的股票池中寻找真正能封板盈利的股票
    - (daily Demo)

#### XXX

In [1]:
### Settings donot modify the code below !!!
import panel
import pandas as pd
import warnings
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
pd.set_option('display.max_rows', 200)
warnings.filterwarnings("ignore")
begDate    = "20170101"
endData    = "20250103"

from busdates import PortableBusDates
from tqdm import tqdm
pbd = PortableBusDates()
tradingdays = pbd.get_range(begDate, endData)
preface_tradingdays = pbd.get_range(pbd.prev_date(begDate, 122), begDate) + tradingdays
def safe_div(x, y):
    return np.where(y<1e-8, np.nan, x/y)
### threshold
thresholduplimt = 0.7  

In [2]:
potential_3s_df = pd.read_parquet("/data/beer2/wensheng/feature_3s_df_univ_isos_delayunivers.parquet", engine="pyarrow")
potential_3s_correct = potential_3s_df[potential_3s_df['high.limit']!=potential_3s_df['see_price']]
potential_3s_correct

,sid,return_open,return_5minopen,return_close,return_5min_1dcont,is_limit_up,time,see_price,buy_price,rob_thre_price,timeHMS,adj.factor.locf,high.limit,bsize1,date
0,1032,-0.026362,-0.017575,0.028120,-0.017575,False,14:06:00,11.28,11.28,11.275,14:05:18,4.586018,11.59,11300,20170104
1,1215,0.018600,0.039825,0.037418,0.039825,True,10:55:00,45.30,45.40,45.281,10:54:48,2.522424,46.55,6100,20170104
2,1389,-0.013995,-0.013995,-0.038627,-0.013995,False,11:28:00,53.30,53.20,53.209,11:28:00,1.683518,54.70,800,20170104
3,1424,0.030711,0.024401,0.007152,0.024401,True,10:48:00,46.90,46.96,46.244,10:47:15,2.022648,47.54,125000,20170104
5,1515,-0.007944,-0.024713,0.007061,-0.024713,False,10:10:00,11.37,11.32,11.362,10:09:24,5.987291,11.68,400,20170104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171402,628,-0.010403,-0.027308,0.046164,-0.027308,False,10:28:00,15.52,15.71,15.485,10:27:03,7.518336,15.92,300,20250103
171403,702,-0.030395,-0.066869,-0.057751,-0.066869,False,09:47:00,6.67,6.68,6.664,09:46:09,2.074096,6.85,38500,20250103
171404,722,0.000000,0.019649,0.014035,0.019649,True,09:46:00,13.88,13.88,13.860,09:45:39,2.099676,14.25,3600,20250103
171406,877,-0.021898,-0.055474,-0.064234,-0.055474,True,13:30:00,6.67,6.67,6.664,13:29:51,4.293559,6.85,109000,20250103


In [7]:
import gc
gc.collect()

2249

In [ ]:

# feature_1min_df = []
# for sampleDate in tqdm(tradingdays):
def process_day(sampleDate):
    # univers = dailyadjust_df.set_index('date').loc[sampleDate]['sid'].unique()
    univers        = potential_3s_correct.set_index('date').loc[sampleDate]['sid'].unique()

    ## intraday_features
    sample_1min_df = panel.sdiv2df(
        panel.read(f"/data/beer1/data/chinaEquityData/panel/1min-lts/1min-{sampleDate}.sdiv").sel(V=slice("close","close"), I = slice("09:20:00", None))
    )

    sample_1min_df = sample_1min_df[sample_1min_df['sid'].isin(univers)]
    sample_1min_df.set_index(['date','sid','time'], inplace=True)
    sample_1min_df['pct_speedmin'] = sample_1min_df[['close']].groupby('sid', group_keys=False).pct_change(periods=4, fill_method=None).shift() * 100
    sample_1min_df['pct_speed1min'] = sample_1min_df[['close']].groupby('sid', group_keys=False).pct_change(periods=1, fill_method=None).shift() * 100
    return sample_1min_df.reset_index()[['pct_speedmin', 'pct_speed1min','time', 'sid', 'date']]
    # feature_1min_df.append(sample_1min_df.reset_index()[['pct_speed1min','time', 'sid', 'date']])
    

# Parallel化处理
feature_1min_df = Parallel(n_jobs=50, verbose=10)(delayed(process_day)(sampleDate) for sampleDate in tqdm(tradingdays[1:]))

# 合并结果
feature_1min_df = pd.concat(feature_1min_df, ignore_index=True)
# feature_1min_df.to_parquet("./uplimit_beer/feature_1min_df_001.parquet", engine="pyarrow", compression="snappy")



[Parallel(n_jobs=50)]: Using backend LokyBackend with 50 concurrent workers.




[Parallel(n_jobs=50)]: Done  13 tasks      | elapsed:   11.2s
[Parallel(n_jobs=50)]: Done  28 tasks      | elapsed:   11.7s
[Parallel(n_jobs=50)]: Done  45 tasks      | elapsed:   12.0s


[Parallel(n_jobs=50)]: Done  62 tasks      | elapsed:   13.8s
[Parallel(n_jobs=50)]: Done  81 tasks      | elapsed:   16.8s
[Parallel(n_jobs=50)]: Done 100 tasks      | elapsed:   18.9s


[Parallel(n_jobs=50)]: Done 121 tasks      | elapsed:   19.6s
[Parallel(n_jobs=50)]: Done 142 tasks      | elapsed:   21.2s


[Parallel(n_jobs=50)]: Done 165 tasks      | elapsed:   23.0s
[Parallel(n_jobs=50)]: Done 188 tasks      | elapsed:   23.8s




[Parallel(n_jobs=50)]: Done 213 tasks      | elapsed:  1.1min
[Parallel(n_jobs=50)]: Done 238 tasks      | elapsed:  1.1min


100%|██████████| 395/395 [01:10<00:00,  5.64it/s]
[Parallel(n_jobs=50)]: Done 265 tasks      | elapsed:  1.2min
[Parallel(n_jobs=50)]: Done 292 tasks      | elap

In [10]:
### load the panel you need here 
# all_daily_data = []
# all_dailyadjust_data = []
def process_daily(sampleDate):
# for sampleDate in tqdm(preface_tradingdays):
    univers          = panel.sdiv2df(panel.read(f"/data/beer2/wensheng/univers_nost/morning/daily/terms.{sampleDate}.sdiv")).set_index('sid').index  # sampleDate next
    daily_date       =  panel.sdiv2df(panel.read(f"/data/nfs0/chinaEquityData/panel/daily/daily-{sampleDate}.sdiv"))
    dailyadjust_date = panel.sdiv2df(panel.read(f"/data/nfs0/chinaEquityData/panel/daily-adjusted-2007/morning/daily-adjusted-{sampleDate}.sdiv"))
    sample_sw2_df             = panel.sdiv2df(panel.read(f"/data/nfs0/chinaEquityData/panel/industry-sw/industry-sw2-{sampleDate}.sdiv"))
    sample_cap_date_df        = panel.sdiv2df(panel.read(f"/data/nfs0/chinaEquityData/panel/cap/cap-{sampleDate}.sdiv")) 
    sample_sw2_df['industry'] = sample_sw2_df[sample_sw2_df.columns[:-3]].idxmax(axis=1)
    # all_daily_data.append(daily_date
    # all_dailyadjust_data.append(dailyadjust_date)
    return daily_date[daily_date['sid'].isin(univers)],  dailyadjust_date[dailyadjust_date['sid'].isin(univers)], sample_sw2_df[['date', 'sid', 'industry']][sample_sw2_df['sid'].isin(univers)], sample_cap_date_df[sample_cap_date_df['sid'].isin(univers)]

results = Parallel(n_jobs=50, verbose=10)(delayed(process_daily)(sampleDate) for sampleDate in tqdm(preface_tradingdays))

all_daily_data_df   = pd.concat([result[0] for result in results], ignore_index=True).set_index(['sid','date'])
dailyadjust_data_df = pd.concat([result[1] for result in results], ignore_index=True).set_index(['sid','date'])
ind_sw2_df          = pd.concat([result[2] for result in results], ignore_index=True).set_index(['sid','date'])
cap_data_df         = pd.concat([result[3] for result in results], ignore_index=True).set_index(['sid','date']) 
# all_daily_data_df   = pd.concat(all_daily_data, ignore_index=True).set_index(['sid','date'])
# dailyadjust_data_df = pd.concat(all_dailyadjust_data, ignore_index=True).set_index(['sid','date'])




[Parallel(n_jobs=50)]: Using backend LokyBackend with 50 concurrent workers.


[Parallel(n_jobs=50)]: Done  13 tasks      | elapsed:    7.7s
[Parallel(n_jobs=50)]: Done  28 tasks      | elapsed:    7.8s
[Parallel(n_jobs=50)]: Done  45 tasks      | elapsed:    7.9s


[Parallel(n_jobs=50)]: Done  62 tasks      | elapsed:   11.4s
[Parallel(n_jobs=50)]: Done  81 tasks      | elapsed:   11.5s
[Parallel(n_jobs=50)]: Done 100 tasks      | elapsed:   11.7s


[Parallel(n_jobs=50)]: Done 121 tasks      | elapsed:   14.8s
[Parallel(n_jobs=50)]: Done 142 tasks      | elapsed:   16.0s


[Parallel(n_jobs=50)]: Done 165 tasks      | elapsed:   18.9s
[Parallel(n_jobs=50)]: Done 188 tasks      | elapsed:   20.0s


[Parallel(n_jobs=50)]: Done 213 tasks      | elapsed:   22.6s
[Parallel(n_jobs=50)]: Done 238 tasks      | elapsed:   23.5s


[Parallel(n_jobs=50)]: Done 265 tasks      | elapsed:   25.4s
[Parallel(n_jobs=50)]: Done 292 tasks      | elapsed:   26.2s


[Parallel(n_jobs=50)]: Done 321 tasks  

In [11]:
all_daily_data_df['close']   = all_daily_data_df['close'] * dailyadjust_data_df['adj.factor.locf']
all_daily_data_df['pct_df']  = all_daily_data_df.groupby('sid')['close'].pct_change()
all_daily_data_df['open']    = all_daily_data_df['open'] * dailyadjust_data_df['adj.factor.locf']
all_daily_data_df['high']    = all_daily_data_df['high'] * dailyadjust_data_df['adj.factor.locf']
all_daily_data_df['low']     = all_daily_data_df['low']  * dailyadjust_data_df['adj.factor.locf']
all_daily_data_df['pct_df']  = all_daily_data_df.groupby('sid')['close'].pct_change()



def calculate_corr(group):
    return group['close'].rolling(window=55, min_periods=35).corr(group['volume']).shift()
dpv_corr = all_daily_data_df.groupby('sid', group_keys=False)[['close','volume']].apply(calculate_corr)

amt75_mean  = all_daily_data_df.groupby('sid', group_keys=False)['volume'].rolling(70).mean().shift().droplevel(level=0)
amt75_std   = all_daily_data_df.groupby('sid', group_keys=False)['volume'].rolling(70).std().shift().droplevel(level=0)
amt75_std[amt75_std<1e-8] = np.nan 
amt75_zsc   = (all_daily_data_df.groupby('sid', group_keys=False)['volume'].shift() - amt75_mean) / amt75_std


ret10_mean  = all_daily_data_df.groupby('sid', group_keys=False)['pct_df'].rolling(10).mean().shift().droplevel(level=0)
ret10_std   = all_daily_data_df.groupby('sid', group_keys=False)['pct_df'].rolling(10).std().shift().droplevel(level=0)
ret10_std[ret10_std<1e-8] = np.nan 
ret10_zsc   = (all_daily_data_df.groupby('sid', group_keys=False)['pct_df'].shift() - ret10_mean) / ret10_std
close_mean = all_daily_data_df.groupby('sid', group_keys=False)['close'].rolling(20).mean().shift().droplevel(level=0)
close_std  = all_daily_data_df.groupby('sid', group_keys=False)['close'].rolling(20).std().shift().droplevel(level=0)
close_std[close_std<1e-8] = np.nan 
prs_zsc    = (all_daily_data_df.groupby('sid', group_keys=False)['close'].shift() - close_mean) / close_std


skew_ret_long     = all_daily_data_df.groupby('sid', group_keys=False)['pct_df'].rolling(75).skew().shift().droplevel(level=0)
skew_ret_longlong = all_daily_data_df.groupby('sid', group_keys=False)['pct_df'].rolling(120).skew().shift().droplevel(level=0)


merged_df = all_daily_data_df[['pct_df']].merge(
    ind_sw2_df[['industry']], left_index=True, right_index=True, how='left'
).merge(
    cap_data_df[['free.mv']], left_index=True, right_index=True, how='left'
)


def calc_industry_weighted_ret(group):
    group = group.dropna(subset=['pct_df'])
    if group.empty:
        return np.nan
    weighted_ret = (group['pct_df'] * group['free.mv']).sum() / group['free.mv'].sum()
    return weighted_ret

industry_weighted_ret = merged_df.groupby(['date', 'industry']).apply(calc_industry_weighted_ret).reset_index(name='industry_weighted_ret')
merged_df  = merged_df.reset_index().merge(
    industry_weighted_ret[['date', 'industry', 'industry_weighted_ret']],
    on=['date', 'industry'], how='left'
)
industry_wavg_df = merged_df[['industry_weighted_ret', 'sid', 'date']].set_index(['sid', 'date'])
industry_wavg_5mean = industry_wavg_df.groupby('sid', group_keys=False)['industry_weighted_ret'].rolling(5).mean().shift().droplevel(level=0)

merged_df['rank'] = merged_df.groupby(['date', 'industry'])['pct_df'].rank(method='min')
valid_stock_count = merged_df.groupby(['date', 'industry'])['pct_df'].transform(lambda x: x.notna().sum())
merged_df['normalized_rank'] = (merged_df['rank'] - 1) / (valid_stock_count - 1)
merged_df['normalized_rank'] = merged_df['normalized_rank'].where(merged_df['pct_df'].notna())
grouprank_ind_ret = merged_df.reset_index()[['normalized_rank', 'sid', 'date']].set_index(['sid', 'date'])
grouprank_ind_ret_prev = grouprank_ind_ret.groupby('sid', group_keys=False)['normalized_rank'].shift()#.droplevel(level=0)
# grouprank_ind_ret_20mean = grouprank_ind_ret.groupby('sid', group_keys=False)['normalized_rank'].rolling(20).mean().shift().droplevel(level=0)

def calculate_corr(group):
    return group['pct_df'].rolling(window=15, min_periods=15).corr(group['industry_weighted_ret']).shift()
groupret_corr = merged_df.set_index(['sid','date']).groupby('sid', group_keys=False).apply(calculate_corr)



merged_df.set_index(['sid','date'], inplace=True)
win = 20
merged_df['industry_weighted_ret_80th'] = merged_df.groupby('sid', group_keys=False)['industry_weighted_ret'].rolling(win, min_periods=win).quantile(0.8).droplevel(level=0)
industry_weighted_ret = merged_df['industry_weighted_ret'].unstack()
pct_df = merged_df['pct_df'].unstack()
industry_weighted_ret_80th = merged_df['industry_weighted_ret_80th'].unstack()
ans_list = []
for i, sampleDate in tqdm(enumerate(pct_df.columns)):
    if i<=win:
        continue
    ans_arr = np.nanstd(np.where(industry_weighted_ret.iloc[:, i-win:i].values > industry_weighted_ret_80th.iloc[:, i-1:i].values, pct_df.iloc[:, i-win:i].values, np.nan), axis=-1)
    ans_list.append(pd.DataFrame(ans_arr, index=pct_df.index, columns=[sampleDate]))
ans_df = pd.concat(ans_list, axis=1)
ans_df.columns.name='date'
top_ind_soilders = ans_df.stack(dropna=False)




































































































































2067it [00:06, 300.29it/s]


In [12]:

## may not in use 
low20   = all_daily_data_df.groupby('sid', group_keys=False)['low'].rolling(20).min().shift().droplevel(level=0)
boundup = (all_daily_data_df.groupby('sid', group_keys=False)['close'].shift() - low20) / low20
prx5_std           = all_daily_data_df.groupby('sid', group_keys=False)['close'].rolling(5).std().shift().droplevel(level=0)
prx100_std         = all_daily_data_df.groupby('sid', group_keys=False)['close'].rolling(120).std().shift().droplevel(level=0)
prxstd_long_short  = prx5_std / prx100_std
amt5_std    = all_daily_data_df.groupby('sid', group_keys=False)['volume'].rolling(5).std().shift().droplevel(level=0)


In [ ]:

result_df = pd.DataFrame({
    'industry_wavg_5mean': industry_wavg_5mean,
    'grouprank_ind_ret_prev': grouprank_ind_ret_prev,
    'groupret_corr': groupret_corr,
    'top_ind_soilders': top_ind_soilders,
    'skew_ret_long': skew_ret_long,
    'skew_ret_longlong': skew_ret_longlong,
    'top_ind_soilders' :top_ind_soilders,
    'amt75_zsc': amt75_zsc,
    'amt75_std': amt75_std,
    'dpv_corr' : dpv_corr, 
    'boundup': boundup,   ## need check
    'prxstd_long_short': prxstd_long_short,
    'amt5_std': amt5_std,
    }
)

result_df      = result_df.loc[pd.IndexSlice[:, begDate:], :].replace(np.inf, np.nan).replace(-np.inf, np.nan)
result_df_rank = result_df.groupby('date', group_keys=False).apply(lambda seq: seq.rank() / pd.notnull(seq).sum())


result_df2 = pd.DataFrame({
    'ret10_zsc': ret10_zsc,
    'ret10_std': ret10_std,
    'prs_zsc': prs_zsc,
    }
)



## may not in use 


# Process_1min_df = []
def calculate_corr(group):
    return group['dclose'].rolling(window=5, min_periods=4).corr(group['volumeTotal']).shift()
# 处理每个sampleDate的函数
# for sampleDate in tqdm(preface_tradingdays):
def process_1min_rev(sampleDate):
    # 读取数据
    univers = potential_3s_correct.set_index('date').loc[sampleDate]['sid'].unique()
    sample_1min_df = panel.sdiv2df(
        panel.read(f"/data/beer1/data/chinaEquityData/panel/1min-lts/1min-{pbd.prev_date(sampleDate)}.sdiv").sel(V =slice("close","volumeTotal"), I = slice("10:00:00","14:30:00"))
    )
    sample_1min_df = sample_1min_df[sample_1min_df['sid'].isin(univers)]
    sample_1min_df.set_index(['sid','time'], inplace=True)

    sample_1min_df['dclose']   = sample_1min_df.groupby('sid', group_keys=False)['close'].shift()
    sample_1min_df['dpv_corr_6min'] = sample_1min_df[['dclose','volumeTotal']].groupby('sid', group_keys=False).apply(calculate_corr).replace(np.inf, np.nan).replace(-np.inf, np.nan)

    agg_sample1min = sample_1min_df.groupby('sid', group_keys=False).agg(
        dpv_corr_6min_mean=('dpv_corr_6min', 'mean'),
    ).reset_index()
    
    # agg_sample1min['dpv_corr_6min_mean'] = agg_sample1min['dpv_corr_6min_mean'].rank() / pd.notnull(agg_sample1min['dpv_corr_6min_mean']).sum()
    agg_sample1min['date'] = sampleDate
    return agg_sample1min
    # Process_1min_df.append(agg_sample1min)
    

Process_1min_df = Parallel(n_jobs=100, verbose=10)(delayed(process_1min_rev)(sampleDate) for sampleDate in tqdm(tradingdays[1:]))
agg_1min_feat_df = pd.concat(Process_1min_df, ignore_index=True)
agg_1min_feat_df.set_index(['sid','date'], inplace=True)
# agg_1min_feat_df['dpv_corr_6min_mean_20avg'] = agg_1min_feat_df.groupby('sid', group_keys=False)['dpv_corr_6min_mean'].rolling(20).mean().droplevel(level=0)
# agg_1min_feat_df['dpv_corr_6min_mean_20std']  = agg_1min_feat_df.groupby('sid', group_keys=False)['dpv_corr_6min_mean'].rolling(20).std().droplevel(level=0)
# agg_1min_feat_df['dpv_corr_6min_mean_ratio'] = agg_1min_feat_df['dpv_corr_6min_mean'] - agg_1min_feat_df['dpv_corr_6min_mean_20avg']
# agg_1min_feat_df['dpv_corr_6min_mean_zsc'] = agg_1min_feat_df['dpv_corr_6min_mean_ratio'] / agg_1min_feat_df['dpv_corr_6min_mean_20std']



feature_all_df = feature_1min_df.set_index(['sid','date','time']).merge(result_df_rank, how='left', left_index=True, right_index=True)
feature_all_df = feature_all_df.merge(result_df2, how='left', left_index=True, right_index=True)
feature_all_df = feature_all_df.merge(agg_1min_feat_df, how='left', left_index=True, right_index=True)
feature_all_df.to_parquet("/data/beer2/wensheng/features_copy/tree_bm_features_isos.parquet", engine="pyarrow", compression="snappy")

In [14]:

# feature_1min_df = []
# for sampleDate in tqdm(tradingdays):
def process_day(sampleDate):
    # univers = dailyadjust_df.set_index('date').loc[sampleDate]['sid'].unique()
    univers        = potential_3s_correct.set_index('date').loc[sampleDate]['sid'].unique()

    ## intraday_features
    sample_1min_df = panel.sdiv2df(
        panel.read(f"/data/beer1/data/chinaEquityData/panel/1min-lts/1min-{sampleDate}.sdiv").sel(V=slice("close","close"), I = slice("09:20:00", None))
    )

    sample_1min_df = sample_1min_df[sample_1min_df['sid'].isin(univers)]
    sample_1min_df.set_index(['date','sid','time'], inplace=True)
    sample_1min_df['pct_speedmin'] = sample_1min_df[['close']].groupby('sid', group_keys=False).pct_change(periods=4, fill_method=None).shift() * 100
    sample_1min_df['pct_speed1min'] = sample_1min_df[['close']].groupby('sid', group_keys=False).pct_change(periods=1, fill_method=None).shift() * 100
    return sample_1min_df.reset_index()[['pct_speedmin', 'pct_speed1min','time', 'sid', 'date']]
    # feature_1min_df.append(sample_1min_df.reset_index()[['pct_speed1min','time', 'sid', 'date']])
    

# Parallel化处理
feature_1min_df = Parallel(n_jobs=50, verbose=10)(delayed(process_day)(sampleDate) for sampleDate in tqdm(tradingdays[1:]))

# 合并结果
feature_1min_df = pd.concat(feature_1min_df, ignore_index=True)
# feature_1min_df.to_parquet("./uplimit_beer/feature_1min_df_001.parquet", engine="pyarrow", compression="snappy")



[Parallel(n_jobs=50)]: Using backend LokyBackend with 50 concurrent workers.






[Parallel(n_jobs=50)]: Done  13 tasks      | elapsed:   50.9s
[Parallel(n_jobs=50)]: Done  28 tasks      | elapsed:   51.8s
[Parallel(n_jobs=50)]: Done  45 tasks      | elapsed:   52.8s


[Parallel(n_jobs=50)]: Done  62 tasks      | elapsed:   53.7s
[Parallel(n_jobs=50)]: Done  81 tasks      | elapsed:   54.6s
[Parallel(n_jobs=50)]: Done 100 tasks      | elapsed:   55.3s


[Parallel(n_jobs=50)]: Done 121 tasks      | elapsed:   56.2s
[Parallel(n_jobs=50)]: Done 142 tasks      | elapsed:   57.1s


[Parallel(n_jobs=50)]: Done 165 tasks      | elapsed:   58.4s
[Parallel(n_jobs=50)]: Done 188 tasks      | elapsed:   59.5s


[Parallel(n_jobs=50)]: Done 213 tasks      | elapsed:  1.0min
[Parallel(n_jobs=50)]: Done 238 tasks      | elapsed:  1.0min


[Parallel(n_jobs=50)]: Done 265 tasks      | elapsed:  1.1min
[Parallel(n_jobs=50)]: Done 292 tasks      | elapsed:  1.1min


[Parallel(n_jobs=50)]: Done 321 tas

In [ ]:
# factor_all_df['date_dt'] = pd.to_datetime(factor_all_df['date'].astype(str), format='%Y%m%d')
# factor_all_df['last_year_uplimit2'] = factor_all_df.groupby('sid').apply(
#     lambda group: group.sort_values('date_dt')
#     .rolling('365D', on='date_dt', closed='left')['continue_limit_num']
#     .sum()
# ).reset_index(level=0, drop=True)
# factor_all_df['last_year_uplimit2'] = factor_all_df['last_year_uplimit2'].fillna(0)